In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS
workspace.default.theme_park_files;

In [0]:
%sql
SHOW VOLUMES IN workspace.default;

In [0]:
%sql
LIST '/Volumes/workspace/default/theme_park_files/';

A. Data ingestion and exploration

01. Load the supplied CSV and JSON datasets using the setup notebook. Display each table and verify its
row count.
02. Inspect the schema of tp_entries_raw. Identify string, integer, and decimal columns.
03. Select entry_id, attraction_id, visit_date, ticket_type, and ticket_price. Display the first 20 rows.
04. Using SQL, find Completed entries where ticket_price is greater than 500.
05. Use DISTINCT and IN to examine ticket types and booking channels.
06. Convert visit_date to a date type and count entries by visit date

In [0]:
%sql
SELECT 'tp_entries_raw' AS table_name, COUNT(*) AS total_rows
FROM workspace.default.tp_entries_raw

UNION ALL

SELECT 'tp_attractions', COUNT(*)
FROM workspace.default.tp_attractions

UNION ALL

SELECT 'tp_maintenance', COUNT(*)
FROM workspace.default.tp_maintenance

UNION ALL

SELECT 'tp_entry_changes', COUNT(*)
FROM workspace.default.tp_entry_changes;

In [0]:
%sql
SELECT * FROM workspace.default.tp_entries_raw;

In [0]:
%sql
SELECT * FROM workspace.default.tp_attractions;

In [0]:
%sql
SELECT * FROM workspace.default.tp_maintenance;

In [0]:
%sql
SELECT * FROM workspace.default.tp_entry_changes;

In [0]:
%sql
DESCRIBE workspace.default.tp_entries_raw;

In [0]:
%sql
select entry_id,attraction_id,visit_date,ticket_type,ticket_price from workspace.default.tp_entries_raw
LIMIT(20);

In [0]:
%sql
select * from workspace.default.tp_entries_raw where status="Completed" and ticket_price>500;

In [0]:
%sql
select distinct booking_channel from workspace.default.tp_entries_raw;

In [0]:
%sql
select distinct ticket_type from workspace.default.tp_entries_raw;


In [0]:
%sql
select * from workspace.default.tp_entries_raw where booking_channel in ('App','Web');

In [0]:
from pyspark.sql.functions import to_date,col,count 
df=spark.table("workspace.default.tp_entries_raw")
df=df.withColumn("visit_date",to_date(col("visit_date"),"yyyy-MM-dd"))
df.groupBy("visit_date").agg(count("entry_id").alias("total_count")).show()

B. Cleaning and transforming with PySpark

07. Find records with missing ticket_price or booking_channel values.
08. Fill missing booking_channel values with Unknown in a new DataFrame; do not change the raw source.
09. Identify records with negative wait_minutes, zero ride_minutes, or nonpositive ticket_price.
10. Find duplicate entry_id values and create a deduplicated DataFrame.
11. Identify entry records whose attraction_id is missing from the attraction master.
12. Create wait_category: Short (0-20), Medium (21-45), Long (above 45). Exclude invalid waits.
13. Join entries with attractions and extract location.zone and features from the nested JSON data.

In [0]:
from pyspark.sql.functions import col
df=spark.table("workspace.default.tp_entries_raw")
df.filter(col("ticket_price").isNull() | col("booking_channel").isNull()).show()

In [0]:
df2=df.fillna({"booking_channel":"Unknown"})
df2.show()

In [0]:
df.filter((col("wait_minutes")<0) | (col("ride_minutes")==0) | (col("ticket_price")<0)).show()

In [0]:
from pyspark.sql.functions import count, col

duplicate_df = df.groupBy("entry_id").agg(
    count("*").alias("total_count")
).filter(col("total_count") > 1)

duplicate_df.show()

In [0]:
deduplicate_df=duplicate_df.dropDuplicates(["entry_id"])
deduplicate_df.show()

In [0]:
entries_df = spark.table("workspace.default.tp_entries_raw")
attractions_df = spark.table("workspace.default.tp_attractions")

missing_attractions_df = entries_df.join(
    attractions_df,
    on="attraction_id",
    how="left_anti"
)

missing_attractions_df.show()

In [0]:
from pyspark.sql.functions import when

df2 = df.filter(col("wait_minutes") >= 0)

df2 = df2.withColumn(
    "wait_category",
    when(col("wait_minutes") <= 20, "Short")
    .when(col("wait_minutes") <= 45, "Medium")
    .otherwise("Long")
)

df2.show()

In [0]:

joined_df = entries_df.join(
    attractions_df,
    on="attraction_id",
    how="inner"
)

In [0]:
result_df = joined_df.select(
    "entry_id",
    "attraction_id",
    "location.zone",
    "features"
)

result_df.show()

C. SQL joins and aggregations

14. Find Completed entries for Standard or Express tickets using WHERE, IN, AND, and BETWEEN.
15. Calculate total valid ticket revenue and completed-entry counts by attraction.
16. Calculate average waiting time by attraction category, excluding invalid waits.
17. Join maintenance records with attractions and calculate completed-maintenance downtime by
attraction.
18. Calculate total completed-entry revenue by park zone using the nested location.zone attribute.


In [0]:
%sql
select * from tp_entries_raw where ticket_type in ("Standard", "Express") and status='Completed'; 

In [0]:
attractions_df.show(5)

In [0]:
entries_df.show(5)

In [0]:
from pyspark.sql.functions import sum

result_df = (
    entries_df
    .filter(
        (col("status") == "Completed") &
        (col("ticket_price") > 0)
    )
    .groupBy("attraction_id")
    .agg(
        sum("ticket_price").alias("total_valid_revenue"),
        count("entry_id").alias("entry_count")
    )
)

result_df.show()

In [0]:
from pyspark.sql.functions import avg
result_df=entries_df.filter(col("wait_minutes")>0)
result_df= result_df.groupBy("attraction_id").agg(avg("wait_minutes").alias("total_wait_minutes")).show()


In [0]:
maintenance_df = spark.table("workspace.default.tp_maintenance")
join_df=attractions_df.join(maintenance_df,on="attraction_id",how="inner")
join_df.show(5)


In [0]:
(join_df.groupBy("attraction_id")
.agg(sum("downtime_minutes").alias("total_downtime_minutes")).show())

In [0]:
result_df = (
    joined_df
    .filter(
        (col("status") == "Completed") &
        (col("ticket_price") > 0)
    )
    .groupBy(col("location.zone").alias("park_zone"))
    .agg(
        sum("ticket_price").alias("total_revenue")
    )
)

result_df.show()

D. Delta table creation and CRUD

19. Create a clean managed Delta table named tp_entries_work using CTAS or a PySpark DataFrame write.
20. Create a second Delta table from a PySpark DataFrame using saveAsTable().
21. Insert two new valid visitor entries into tp_entries_work using SQL.
22. Update the status and ticket price of an existing entry in tp_entries_work.
23. Delete a Cancelled entry from tp_entries_work and verify the result.
24. Use DESCRIBE EXTENDED to identify the working table type and location; explain managed versus
external storage

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.tp_entries_work
USING DELTA
AS
SELECT *
FROM workspace.default.tp_entries_raw
WHERE ticket_price > 0;

In [0]:
df = spark.table("workspace.default.tp_entries_work")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.tp_entries_backup")

In [0]:
%sql
INSERT INTO workspace.default.tp_entries_work
(
    entry_id,
    visitor_id,
    attraction_id,
    visit_date,
    entry_time,
    wait_minutes,
    ride_minutes,
    ticket_type,
    ticket_price,
    status,
    booking_channel
)
VALUES
(900001, 'V9001', 'A001', '11-11-2026', '10:30:00', 15, 5, 'Adult', 750.0, 'Completed', 'Online'),
(900002, 'V9002', 'A002', '11-11-2026', '11:00:00', 25, 8, 'Child', 400.0, 'Completed', 'Counter');

In [0]:
%sql
UPDATE tp_entries_work
SET
    status = 'Completed',
    ticket_price = 850.0
WHERE entry_id = 900001;

In [0]:
%sql
DELETE FROM tp_entries_work
WHERE status = 'Cancelled';

In [0]:
%sql
SELECT *
FROM tp_entries_work
WHERE status = 'Cancelled';

In [0]:
%sql
DESCRIBE EXTENDED tp_entries_work;

E. MERGE / UPSERT and SCD

25. Inspect tp_entry_changes and separate existing entry IDs from new IDs.
26. Use SQL MERGE to update matching entries and insert new entries into tp_entries_work (SCD Type 1).
27. Verify which entry IDs were updated and which were inserted.
28. Create a separate Delta table and perform the same MERGE using the PySpark DeltaTable API.
29. Create an attraction-capacity history table with valid_from, valid_to, and is_current columns.
30. Implement an SCD Type 2 change for one attraction capacity; query both its historical and current
rows.

In [0]:
work_df = spark.table("workspace.default.tp_entries_work")
entry_df=spark.table("workspace.default.tp_entry_changes")
existing_ids = entry_df.join(
    work_df,
    on="entry_id",
    how="inner"
)

new_ids = entry_df.join(
    work_df,
    on="entry_id",
    how="left_anti"
)

print("Existing Entry IDs:")
existing_ids.select("entry_id").show()

print("New Entry IDs:")
new_ids.select("entry_id").show()

In [0]:
%sql
MERGE INTO workspace.default.tp_entries_work AS target
USING workspace.default.tp_entry_changes AS source
ON target.entry_id = source.entry_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
print("Updated Entry IDs:")
existing_ids.select("entry_id").show()

print("Inserted Entry IDs:")
new_ids.select("entry_id").show()

In [0]:
from delta.tables import DeltaTable
entry_df = spark.table("workspace.default.tp_entries_raw")
entry_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.tp_entries_merge_api")

target = DeltaTable.forName(spark,"workspace.default.tp_entries_merge_api")

source = spark.table("workspace.default.tp_entry_changes")


target.alias("target").merge(
    source.alias("source"),
    "target.entry_id = source.entry_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

spark.table("workspace.default.tp_entries_merge_api").show()

In [0]:
from pyspark.sql.functions import current_date, lit

attractions_df = spark.table("workspace.default.tp_attractions")

capacity_history_df = (
    attractions_df
    .select("attraction_id", "attraction_name", "capacity_per_hour")
    .withColumn("valid_from", current_date())
    .withColumn("valid_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

capacity_history_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.tp_attraction_capacity_history")

spark.table("workspace.default.tp_attraction_capacity_history").show()

In [0]:
df = spark.table("workspace.default.tp_attraction_capacity_history")

record = df.first()
attraction_id = record["attraction_id"]

target = DeltaTable.forName(spark,"workspace.default.tp_attraction_capacity_history")

target.update(condition=(col("attraction_id") == attraction_id) & (col("is_current") == True),
    set={
        "valid_to": current_date(),
        "is_current": lit(False)
    }
)
new_df = spark.createDataFrame(
    [(attraction_id, record["attraction_name"], record["capacity_per_hour"] + 20)],
    ["attraction_id", "attraction_name", "capacity_per_hour"]
)

new_df = new_df.withColumn("valid_from", current_date()) \
    .withColumn("valid_to", lit(None).cast("date")) \
    .withColumn("is_current", lit(True))

new_df.write.format("delta").mode("append").saveAsTable(
    "workspace.default.tp_attraction_capacity_history"
)

spark.table("workspace.default.tp_attraction_capacity_history") \
    .filter(col("attraction_id") == attraction_id) \
    .orderBy("valid_from") \
    .show(truncate=False)

F. Delta internals, Time Travel, and optimization

31. Use DESCRIBE DETAIL and DESCRIBE HISTORY on tp_entries_work. Record the storage format and
operations.
32. Find the table version immediately before your MERGE and query it using VERSION AS OF.
33. Compare one updated entry before and after the MERGE.
34. Run OPTIMIZE, then compare the file count before and after. Explain why small tables may not change.
35. Run ZORDER BY (attraction_id) if your table is not liquid clustered, and inspect the history.
36. Explain VACUUM retention and perform a safe dry run if supported; do not delete historical files.

In [0]:
%sql
DESCRIBE DETAIL tp_entries_work;

In [0]:
%sql
DESCRIBE HISTORY tp_entries_work;

In [0]:
%sql
SELECT * FROM tp_entries_work VERSION AS OF 5;

In [0]:
%sql
SELECT entry_id, ticket_price, status
FROM tp_entries_work VERSION AS OF 5
WHERE entry_id = 1040;



In [0]:
%sql
SELECT entry_id, ticket_price, status
FROM tp_entries_work VERSION AS OF 6
WHERE entry_id = 1040;

In [0]:
%sql
DESCRIBE DETAIL tp_entries_work;

In [0]:
%sql
OPTIMIZE tp_entries_work;

In [0]:
%sql
DESCRIBE DETAIL tp_entries_work;

In [0]:
%sql
OPTIMIZE tp_entries_work
ZORDER BY (attraction_id);

In [0]:
%sql
describe history tp_entries_work;

In [0]:
%sql
VACUUM tp_entries_work DRY RUN;

G. Views and Parquet

37. Create a permanent view of valid Completed entries joined with attraction names.
38. Create a temporary view of entries with wait_minutes greater than 45.
39. Create and query a global temporary view if supported by your compute.
40. Write a DataFrame to Parquet in an approved Volume or external location; read it back and inspect its
schema.
41. Convert that Parquet data to a new Delta table by reading it and writing Delta; explain what the Delta
transaction log adds

In [0]:
%sql
CREATE OR REPLACE VIEW completed_entries_view AS
SELECT
    entries.entry_id,
    entries.attraction_id,
    attractions.attraction_name,
    entries.visit_date,
    entries.ticket_type,
    entries.ticket_price,
    entries.status
FROM tp_entries_work entries
INNER JOIN tp_attractions attractions
ON entries.attraction_id = attractions.attraction_id
WHERE entries.status = 'Completed'
AND entries.ticket_price > 0;

In [0]:
%sql
SELECT * FROM completed_entries_view;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW long_wait_entries AS
SELECT *
FROM tp_entries_work
WHERE wait_minutes > 45;

In [0]:
%sql
SELECT * FROM long_wait_entries;

In [0]:
%sql
CREATE OR REPLACE GLOBAL TEMP VIEW global_long_wait_entries AS
SELECT *
FROM tp_entries_work
WHERE wait_minutes > 45;

In [0]:
%sql
SELECT *
FROM global_temp.global_long_wait_entries;

In [0]:
df = spark.table("tp_entries_work")

df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/theme_park_files/tp_entries_parquet")

In [0]:
parquet_df = spark.read \
    .format("parquet") \
    .load("/Volumes/workspace/default/theme_park_files/tp_entries_parquet")

parquet_df.printSchema()
parquet_df.show(5)

In [0]:
parquet_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("tp_entries_from_parquet")

In [0]:
%sql
SELECT * FROM tp_entries_from_parquet;

In [0]:
%sql
DESCRIBE DETAIL tp_entries_from_parquet;

Parquet stores the actual data files. Delta adds a transaction log (_delta_log) that keeps track of changes made to the table.

It provides:
- ACID transactions – reliable data updates.
- Time Travel – access previous table versions.
- Schema enforcement – checks the data structure.
- History – tracks table operations.

H. Structured Streaming

42. Create a streaming DataFrame from Spark rate with five rows per second.
43. Create visitor_event_id and simulated_wait_minutes columns from the streaming value.
44. Filter the stream to retain simulated waits above 30 minutes.
45. Aggregate streaming events into two wait categories, display results with a supported sink, and stop
the query.

In [0]:
stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

stream_df.printSchema()

In [0]:
from pyspark.sql.functions import col

stream_df = stream_df.withColumn(
    "visitor_event_id", col("value")
).withColumn(
    "simulated_wait_minutes", (col("value") % 60)
)

stream_df.printSchema()

In [0]:
filtered_stream_df = stream_df.filter(
    col("simulated_wait_minutes") > 30
)

filtered_stream_df.printSchema()

In [0]:
from pyspark.sql.functions import when,count
category_stream_df = filtered_stream_df.withColumn(
    "wait_category",
    when(col("simulated_wait_minutes") <= 45, "Medium")
    .otherwise("Long")
)

result_df = category_stream_df.groupBy("wait_category").agg(
    count("*").alias("total_events")
)

query = result_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .start()

query.awaitTermination(30)

query.stop()

I. Lakeflow / DLT and incremental loading

46. Create a SQL Lakeflow pipeline with a Bronze streaming table reading append-only tp_entries_raw.
47. Create a Silver streaming table that retains valid Completed entries and calculates ticket revenue.
48. Create a Gold materialized view showing entry counts and revenue by attraction; run and inspect the
pipeline graph.
49. Create a separate Python Lakeflow pipeline for the same Bronze-Silver-Gold flow using distinct output
names.
50. Append park_entries_incremental.csv to tp_entries_raw only after the first pipeline run; rerun the
pipeline and verify the Gold totals.

In [0]:
incremental_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/theme_park_files/park_entries_incremental.csv")

incremental_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("tp_entries_raw")

In [0]:
%sql
SELECT *
FROM tp_entries_gold
ORDER BY attraction_id;

In [0]:
%sql
SELECT *
FROM tp_entries_gold_python
ORDER BY attraction_id;